# portfolio_cdi_bal 

Migrated from onprem: `[PRC].[operation].[CalcCustomerBalances]`


---

## Logic Summary
1. Get reference date (latest `BusinessDate` from `primaryactive.exposure`)
2. Filter active portfolios (`unit_code_profits IS NOT NULL AND PortfolioId < 600`)
3. Calculate per-parent aggregated balances in EUR using `dexposure` + `converttoeur()`, grouped by `uc_id` = `COALESCE(PartyTaxRegistrationNumber, CDI)` (equivalent of `ParentPartyId` on-prem), then fan out to each individual `CDI` of the parent (one output row per `(PortfolioId, CDI)`)
4. Safety check: if no data, stop execution
5. Build `portfolio_cdi_bal` with initial balance override from `initializationexposureinfo` (joined via `dexposure.ExposureKey`)
6. Calculate settlements from `dsettlementexposure`: resolve each settling party's parent (`uc_id`) directly via `party_tax` (CDI→TaxRegistrationNumber, = on-prem `Party.ParentPartyId`) — NOT via `cdi_map`, since a guarantor/co-owner settling party may be absent from dexposure. Aggregate per `(portfolio, uc_id)` and merge on `(portfolio_code, uc_id)`, applying the amount only to parents present in the output (on-prem MERGE `when matched`)
7. Merge settlements into final result and write to target table

## Key Design Decisions
- **uc_id vs CDI (two distinct columns, fan-out):** On-prem `CalcCustomerBalances` aggregates balances at PARENT grain (`uc_id` = `DWH.prt.Party.ParentPartyId`, `nvarchar` in the target), then in the final INSERT fans out to every individual CDI of that parent (join Party on ParentPartyId, pull `f.CDI` from PartyReferenceCodes). So `uc_id` is the coarse parent key and `CDI` is the finer per-party key, with balances repeated across each CDI. On-prem grain (2026-10-30): 390,430 rows, 390,430 distinct CDI, 322,498 distinct uc_id.
- **Cloud substitution:** `uc_id` = `COALESCE(PartyTaxRegistrationNumber, CDI)` (no DWH ParentPartyId in cloud); `CDI` = the individual `dexposure.CDI`. Verified against cloud (2026-10-30): ~322,432 distinct uc_id (vs on-prem 322,498, 0.02%) and ~385,324 (portfolio, CDI) rows (vs 390,430, ~1.3% — the accepted ~36K party-coverage gap plus dexposure day-lag). Balances summed once per (PortfolioId, uc_id), fanned out to each CDI via `cdi_map`. IT confirmed the alphanumeric string domain for `uc_id`/`CDI` is acceptable.
- **Init balance override:** `{env}-cdh.reference.initializationexposureinfo` joined via `dexposure.ExposureKey`, aggregated to uc_id+Portfolio (parent) level, overrides `initial_assign_bal_EUR` when available
- **Settlement grouping:** The settling party on `dsettlementexposure` may be a guarantor/co-owner not present in `dexposure`. Its parent (`uc_id`) is resolved directly from `party_tax` (matching on-prem `Party.ParentPartyId`), so such settlements are not dropped (review #11). Amounts are merged on `(portfolio_code, uc_id)` and applied only to parents already present in the output.

## Table Mapping (onprem → cloud)

| onprem | cloud |
|---|---|
| `dwh.ref.Portfolio` | `{env}-cdh.primaryactive.portfolio` |
| `DWH.prt.Party` + `DWH.ast.Loan` + `DWH.ast.LoanFinancials` | `{env}-cdh.derivative.dexposure` |
| `DWH.adm.SGKCurrency` + `dbo.convertToEuro()` | `{env}-cdh.derivative.converttoeur()` UDF |
| `DWH.adm.SourceExtraction` | `MAX(BusinessDate)` from `primaryactive.exposure` |
| `dwh.adm.PartyReferenceCodes` (CDI) | `CDI` column in `dexposure` |
| `DWH.prt.Party.ParentPartyId` (on-prem `uc_id`, int) | `COALESCE(PartyTaxRegistrationNumber, CDI)` in `dexposure` (cloud `uc_id`, string) |
| `prc.operation.OrbitInitBalance` | `{env}-cdh.reference.initializationexposureinfo` (joined via `ExposureKey`) |
| Settlement tables (`Asset` + `AssetXPartyRltp` + ...) | `{env}-cdh.derivativeactive.dsettlementexposure` | ParticipationPercentage populated for all sources on a unified 0-100 percent scale (divide by 100). |
| `DWH.cls.Classification` (Primary/N/A filter) | `IsActiveFlag` in `dsettlementexposure` |

## Initialization

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sum as spark_sum,
    max as spark_max,
    when
)
from datetime import datetime

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

# Environment prefix (qa / p / d ...), same pattern as platforms/dtransfer_*.
# No hardcoded catalog names anywhere in this notebook.
env = aws_utils.get_env_prefix()

# -- CONFIGURATION --
TARGET_CATALOG = f"{env}-cdh-delivery"
TARGET_SCHEMA  = "it"
TARGET_TABLE   = "portfolio_cdi_bal"

target_fqn = f"`{TARGET_CATALOG}`.`{TARGET_SCHEMA}`.`{TARGET_TABLE}`"

print(f"env: {env} | Target table: {target_fqn}")

## Loading Tables

In [0]:
# Step 1: Get reference date and portfolio

reference_date = spark.sql(f"""
    SELECT MAX(BusinessDate) AS datadate
    FROM `{env}-cdh`.primaryactive.exposure
    WHERE SourceSystem = 'profits'
""")

reference_date.createOrReplaceTempView("reference_date")

datadate = reference_date.collect()[0]["datadate"]
print(f"Reference date: {datadate}")

portfolio = spark.sql(f"""
    SELECT p.PortfolioId, p.PortfolioName, p.Prefix, p.processSettlementFlag
    FROM `{env}-cdh`.primaryactive.portfolio p
    WHERE p.unit_code_profits IS NOT NULL
      AND p.PortfolioId < 600
      AND p.BusinessDate = '{datadate}'
""")

portfolio.createOrReplaceTempView("portfolio")

print(f"Active portfolios: {portfolio.count()}")

#Base Table

In [0]:
# Step 2: Base balance aggregation with EUR conversion
#
# On-prem CalcCustomerBalances aggregates balances at PARENT grain
# (c.ParentPartyId -> uc_id), then in the final INSERT fans the result out to
# every individual CDI belonging to that parent. So uc_id is the coarse parent
# key and CDI is the finer per-party key, with the parent's balances repeated
# across each CDI.
#
# Cloud equivalent: uc_id = COALESCE(NULLIF(TRIM(TaxRegistrationNumber),''), CDI).
# The tax number is resolved from `primary.party` (NOT the dexposure tax column,
# which is NULL for ~3,063 CDIs whose party row carries a tax number).
#
# party_tax: CDI -> non-blank tax, run-date scoped and IDENTICAL to
# dtransfer_Customer.party_tax (review #2/#4): prefer a non-blank tax number,
# then source priority (profits>qca>other), then latest BusinessDate. Blank/
# tax numbers are normalised to NULL (review #3) so they fall back to CDI instead
# of collapsing many customers into one empty parent key.

party_tax = spark.sql(f"""
    SELECT CDI, TaxRegistrationNumber
    FROM (
        SELECT CDI,
               NULLIF(TRIM(TaxRegistrationNumber), '') AS TaxRegistrationNumber,
               ROW_NUMBER() OVER (
                   PARTITION BY CDI ORDER BY
                       CASE WHEN NULLIF(TRIM(TaxRegistrationNumber), '') IS NOT NULL
                            THEN 0 ELSE 1 END,
                       CASE WHEN SourceSystem='profits' THEN 1
                            WHEN SourceSystem='qca' THEN 2 ELSE 3 END,
                       BusinessDate DESC,
                       NULLIF(TRIM(TaxRegistrationNumber), '')
               ) AS _rn
        FROM `{env}-cdh`.primary.party
        WHERE BusinessDate = '{datadate}'
    ) t
    WHERE _rn = 1
""")
# Review #11: materialise party_tax once so base, cdi_map, the init-balance
# subquery and settlements all read the SAME resolved CDI->tax mapping (a lazy
# re-evaluation with a non-total order could otherwise disagree and break the
# balance reconciliation non-deterministically).
party_tax = party_tax.cache()
_party_tax_count = party_tax.count()  # force materialisation
party_tax.createOrReplaceTempView("party_tax")

# Review #6: an empty/stale primary.party snapshot for {datadate} would make every
# COALESCE(tax, CDI) degrade to CDI, collapsing uc_id to one parent per CDI -- a
# silent grain change that passes all four pre-write guards. Fail fast if party_tax
# resolved nothing.
assert _party_tax_count > 0, (
    f"party_tax is empty for datadate={datadate}: primary.party likely not loaded "
    f"for this date. Refusing to run (uc_id would collapse to CDI)."
)

base = spark.sql(f"""
    SELECT
        pf.PortfolioId,
        '{datadate}' AS datadate,
        COALESCE(pt.TaxRegistrationNumber, e.CDI) AS uc_id,
        SUM(`{env}-cdh`.derivative.converttoeur(
            e.Currency, e.OutstandingTotalBalance, '{datadate}'
        )) AS Total_balance_EUR,
        SUM(`{env}-cdh`.derivative.converttoeur(
            e.Currency, e.OnBalance, '{datadate}'
        )) AS on_balance,
        SUM(`{env}-cdh`.derivative.converttoeur(
            e.Currency, e.OffBalance, '{datadate}'
        )) AS off_balance,
        SUM(`{env}-cdh`.derivative.converttoeur(
            e.Currency, e.InitialAssignmentAmount, '{datadate}'
        )) AS initial_assign_bal_EUR
    FROM portfolio pf
    JOIN `{env}-cdh`.derivative.dexposure e
        ON e.PortfolioName = pf.PortfolioName
       AND e.BusinessDate  = '{datadate}'
    LEFT JOIN party_tax pt
        ON pt.CDI = e.CDI
    WHERE e.CDI IS NOT NULL
    GROUP BY pf.PortfolioId, COALESCE(pt.TaxRegistrationNumber, e.CDI)
""")
base.createOrReplaceTempView("base")

# Fan-out map: one row per (PortfolioId, uc_id parent, individual CDI) -- the
# cloud equivalent of on-prem's final-INSERT explosion of each parent into its
# member CDIs. uc_id derived identically to base.
cdi_map = spark.sql(f"""
    SELECT DISTINCT
        pf.PortfolioId,
        COALESCE(pt.TaxRegistrationNumber, e.CDI) AS uc_id,
        e.CDI                                     AS CDI
    FROM portfolio pf
    JOIN `{env}-cdh`.derivative.dexposure e
        ON e.PortfolioName = pf.PortfolioName
       AND e.BusinessDate  = '{datadate}'
    LEFT JOIN party_tax pt
        ON pt.CDI = e.CDI
    WHERE e.CDI IS NOT NULL
""")
cdi_map.createOrReplaceTempView("cdi_map")

print(f"Base rows (parent grain): {base.count()}")
print(f"CDI map rows (portfolio, CDI): {cdi_map.count()}")

# Uniqueness guard: each (PortfolioId, CDI) must map to exactly one uc_id.
_dup_cdi = spark.sql("""
    SELECT PortfolioId, CDI FROM cdi_map
    GROUP BY PortfolioId, CDI HAVING COUNT(DISTINCT uc_id) > 1
""").count()
assert _dup_cdi == 0, f"cdi_map: {_dup_cdi} (PortfolioId, CDI) pairs map to >1 uc_id"

# Review #6: parent roll-up sanity. Healthy ratio distinct(uc_id)/distinct(CDI)
# is ~0.84 (parents group multiple CDIs). If primary.party is empty/stale, uc_id
# collapses to CDI and the ratio jumps to ~1.0. Fail above 0.95 (generous headroom
# over the ~0.84 baseline) so a silent grain collapse cannot be delivered.
_ratio_row = spark.sql("""
    SELECT COUNT(DISTINCT uc_id) AS n_uc, COUNT(DISTINCT CDI) AS n_cdi FROM cdi_map
""").collect()[0]
_rollup_ratio = (_ratio_row['n_uc'] / _ratio_row['n_cdi']) if _ratio_row['n_cdi'] else 1.0
print(f"parent roll-up ratio uc_id/CDI = {_rollup_ratio:.4f} "
      f"({_ratio_row['n_uc']:,} parents / {_ratio_row['n_cdi']:,} CDIs)")
assert _rollup_ratio <= 0.95, (
    f"Parent roll-up collapsed: uc_id/CDI ratio {_rollup_ratio:.4f} > 0.95 "
    f"(baseline ~0.84). primary.party is likely empty/stale for datadate={datadate}; "
    f"uc_id has degraded to CDI. Refusing to deliver at the wrong grain."
)


#Checks

In [0]:
# Step 3: Safety check

base_count = spark.sql("SELECT COUNT(1) AS cnt FROM base").collect()[0]["cnt"]
print(f"Base table row count: {base_count}")

if base_count == 0:
    dbutils.notebook.exit("No data retrieved from base calculation. Keeping previous data. Exiting.")

## portfolio_cdi_bal 

In [0]:
# Step 4: Build the portfolio_cdi_bal result
#
# Join parent-grain balances (base) to the fan-out map (cdi_map) so the result
# has one row per (PortfolioId, individual CDI), carrying uc_id = parent key and
# CDI = the individual party CDI. Balances and init-balance are keyed on
# (PortfolioId, uc_id) and therefore repeat across each CDI of a parent, exactly
# like on-prem.

portfolio_cdi_bal = spark.sql(f"""
    SELECT
        b.PortfolioId                        AS portfolio_code,
        b.datadate                           AS data_date,
        m.uc_id                              AS uc_id,
        m.CDI                                AS CDI,
        b.Total_balance_EUR,
        b.on_balance                         AS on_balance_EUR,
        b.off_balance                        AS off_balance_EUR,
        CASE
            WHEN ib.Init_balance_EUR IS NOT NULL THEN ib.Init_balance_EUR
            ELSE b.initial_assign_bal_EUR
        END                                  AS initial_assign_bal_EUR,
        CAST(NULL AS DOUBLE)                 AS bal_over_settlement
    FROM base b
    JOIN cdi_map m
        ON  m.PortfolioId = b.PortfolioId
       AND m.uc_id       = b.uc_id
    LEFT JOIN (
        SELECT
            COALESCE(pt.TaxRegistrationNumber, e.CDI) AS uc_id,
            pf.PortfolioId,
            SUM(i.InitialAssignmentBalance) AS Init_balance_EUR
        FROM `{env}-cdh`.reference.initializationexposureinfo i
        JOIN `{env}-cdh`.derivative.dexposure e
            ON i.ExposureKey = e.ExposureKey
           AND e.BusinessDate = '{datadate}'
        JOIN portfolio pf
            ON e.PortfolioName = pf.PortfolioName
        LEFT JOIN party_tax pt
            ON pt.CDI = e.CDI
        GROUP BY COALESCE(pt.TaxRegistrationNumber, e.CDI), pf.PortfolioId
    ) ib
        ON ib.uc_id = b.uc_id
       AND ib.PortfolioId = b.PortfolioId
""")

portfolio_cdi_bal.createOrReplaceTempView("portfolio_cdi_bal")


## Settlement Fields

In [0]:
# Step 5: Calculate settlement amounts per parent (uc_id), per portfolio
#
# Source: derivativeactive.dsettlementexposure (NOT primaryactive.settlementexposure).
# The primaryactive table leaves ParticipationPercentage NULL for profits (QC-
# migrated) settlements; the derivative dsettlementexposure populates it for every
# row. On-prem's settlement participation came mostly from QC/Process (frozen
# post-decommission); in cloud those settlements are profits-sourced and only
# dsettlementexposure carries the computed participation.
#
# Scaling: this notebook REQUIRES ParticipationPercentage on a single 0-100 percent
# scale for all sources (divide by 100). A forthcoming dsettlementexposure change
# unifies qca and profits to 0-100; until it is deployed, profits rows are on a 0-1
# scale and this code must NOT run in prod. The validation below enforces the
# contract: it fails the run on NULL, negative, or out-of-0-100 participation
# within the perimeter, per source (review #1/#7), so a mixed-scale or incomplete
# snapshot can never be delivered silently.
#
# Parent resolution (review #11): the settling party (se.CDI) may be a
# guarantor/co-owner absent from dexposure, so resolve uc_id directly from
# party_tax (CDI -> TaxRegistrationNumber), matching on-prem's grouping by
# Party.ParentPartyId. Step 6 applies the amount only to parents present in the
# output (on-prem MERGE 'when matched').
#
# Flag notes: IsActiveFlag is INT (= 1); processSettlementFlag is BOOLEAN (= true).

# --- participation contract validation (per source) ---
_part_check = spark.sql(f"""
    SELECT se.SourceSystem,
           COUNT(*) AS rows,
           SUM(CASE WHEN se.ParticipationPercentage IS NULL THEN 1 ELSE 0 END) AS null_cnt,
           SUM(CASE WHEN se.ParticipationPercentage < 0 THEN 1 ELSE 0 END) AS neg_cnt,
           MIN(se.ParticipationPercentage) AS min_pp,
           MAX(se.ParticipationPercentage) AS max_pp
    FROM `{env}-cdh`.derivativeactive.dsettlementexposure se
    JOIN portfolio pf
        ON pf.PortfolioName = se.PortfolioName
    WHERE se.BusinessDate = '{datadate}'
      AND se.IsActiveFlag  = 1
      AND pf.processSettlementFlag = true
      AND se.CDI IS NOT NULL
    GROUP BY se.SourceSystem
""").collect()

for r in _part_check:
    print(f"participation[{r['SourceSystem']}]: rows={r['rows']:,} null={r['null_cnt']} "
          f"neg={r['neg_cnt']} min={r['min_pp']} max={r['max_pp']}")
    assert r["null_cnt"] == 0, (
        f"dsettlementexposure.ParticipationPercentage has {r['null_cnt']} NULLs for "
        f"source {r['SourceSystem']} in perimeter; settlement amounts would be understated."
    )
    assert r["neg_cnt"] == 0, (
        f"dsettlementexposure.ParticipationPercentage has {r['neg_cnt']} negative "
        f"values for source {r['SourceSystem']}; settlement amounts would be wrong."
    )
    assert (r["max_pp"] is None) or (float(r["max_pp"]) <= 100.0), (
        f"dsettlementexposure.ParticipationPercentage max={r['max_pp']} > 100 for source "
        f"{r['SourceSystem']}: unexpected scale (expected unified 0-100)."
    )
    # Guard against the pre-unification 0-1 scale: if EVERY value is <=1 the source
    # is still on the fraction scale and /100 would understate by 100x.
    assert (r["max_pp"] is None) or (float(r["max_pp"]) > 1.0), (
        f"dsettlementexposure.ParticipationPercentage max={r['max_pp']} <= 1 for source "
        f"{r['SourceSystem']}: looks like the 0-1 scale, not the required unified 0-100 "
        f"(the scaling unification is not deployed yet). Refusing to run."
    )

settlements = spark.sql(f"""
    SELECT
        pf.PortfolioId,
        COALESCE(pt.TaxRegistrationNumber, se.CDI)            AS uc_id,
        SUM((se.ParticipationPercentage / 100.0)
            * se.SettlementAmount)                            AS settlement_amount
    FROM `{env}-cdh`.derivativeactive.dsettlementexposure se
    JOIN portfolio pf
        ON pf.PortfolioName = se.PortfolioName
    LEFT JOIN party_tax pt
        ON pt.CDI = se.CDI
    WHERE se.BusinessDate = '{datadate}'
      AND se.IsActiveFlag  = 1
      AND pf.processSettlementFlag = true
      AND se.CDI IS NOT NULL
    GROUP BY pf.PortfolioId, COALESCE(pt.TaxRegistrationNumber, se.CDI)
""")
settlements.createOrReplaceTempView("settlements")
print(f"Settlement rows (parent grain): {settlements.count()}")

#Final Table

In [0]:
# Step 6: Combine base balances with settlement amounts

portfolio_cdi_bal_final = spark.sql("""
    SELECT
        b.portfolio_code,
        b.data_date,
        b.uc_id,
        b.CDI,
        b.Total_balance_EUR,
        b.on_balance_EUR,
        b.off_balance_EUR,
        -- Column order matches on-prem PRC.operation.portfolio_cdi_bal:
        -- bal_over_settlement precedes initial_assign_bal_EUR.
        s.settlement_amount                  AS bal_over_settlement,
        b.initial_assign_bal_EUR
    FROM portfolio_cdi_bal b
    LEFT JOIN settlements s
        ON  s.PortfolioId = b.portfolio_code
        AND s.uc_id       = b.uc_id
""")

portfolio_cdi_bal_final.createOrReplaceTempView("portfolio_cdi_bal_final")

print(f"Final result rows: {portfolio_cdi_bal_final.count()}")

## Write Table

In [0]:
# Step 7: Write final result to target Delta table
# Original target: [PRC].[operation].[portfolio_cdi_bal]
df_final = spark.sql("SELECT * FROM portfolio_cdi_bal_final")

# ==================== Pre-write safety guards ====================
# The write is an unconditional overwrite, so a silently empty/partial/inflated
# result would replace a good regulatory table. Guards RAISE (fail the task) on
# rejection -- dbutils.notebook.exit() would terminate as SUCCESS and let the
# orchestration treat a rejected delivery as a good refresh (review #5).
final_count = df_final.count()

# Guard 1: never overwrite with an empty result.
if final_count == 0:
    raise ValueError(
        "portfolio_cdi_bal: final result is empty after fan-out; refusing to "
        "overwrite the target. Previous data is preserved."
    )

# Guard 2: balance reconciliation -- the fan-out must neither drop nor double a
# parent's balance: parent-grain total (sum once per uc_id in `base`) must equal
# the fanned output rolled back to one value per (portfolio, uc_id).
parent_total = spark.sql(
    "SELECT SUM(Total_balance_EUR) AS t FROM base"
).collect()[0]["t"] or 0.0
fanned_total = spark.sql("""
    SELECT SUM(t) AS t FROM (
        SELECT portfolio_code, uc_id, MAX(Total_balance_EUR) AS t
        FROM portfolio_cdi_bal_final
        GROUP BY portfolio_code, uc_id
    )
""").collect()[0]["t"] or 0.0
recon_diff = abs(float(parent_total) - float(fanned_total))
assert recon_diff < 1.0, (
    f"Balance reconciliation failed: parent-grain total {parent_total} != "
    f"fanned-out per-(portfolio,uc_id) total {fanned_total} (diff {recon_diff}). "
    f"Fan-out dropped or duplicated a parent's balance."
)

# Guard 2b: row inflation -- every output row must be a distinct (portfolio, CDI).
# Guard 2 nets to zero under duplicate CDI rows (MAX per parent), so check the
# grain explicitly (review #11).
distinct_pc = spark.sql(
    "SELECT COUNT(*) AS c FROM (SELECT DISTINCT portfolio_code, CDI "
    "FROM portfolio_cdi_bal_final)"
).collect()[0]["c"]
assert distinct_pc == final_count, (
    f"Row inflation: {final_count:,} rows but only {distinct_pc:,} distinct "
    f"(portfolio_code, CDI) -- duplicate CDI rows in the output."
)

# Guard 3: day-over-day row-count floor. Raise if the new result is >25% below
# the currently delivered table (silent coverage drop). A missing target is a
# genuine first run; ANY other read failure must stop the write, so narrow the
# handler to table-not-found only (review #5).
from pyspark.sql.utils import AnalysisException
prev_count = None
try:
    prev_count = spark.table(target_fqn).count()
except AnalysisException as e:
    # Only a missing TARGET TABLE is a legitimate first run; schema/catalog
    # errors and any other read failure must stop the write (review #19).
    ec = getattr(e, 'getErrorClass', lambda: '')() or ''
    if ec == 'TABLE_OR_VIEW_NOT_FOUND' or 'TABLE_OR_VIEW_NOT_FOUND' in str(e):
        prev_count = None  # first run, no previous data to compare
    else:
        raise

if prev_count is not None and prev_count > 0 and final_count < prev_count * 0.75:
    raise ValueError(
        f"portfolio_cdi_bal row-count floor: new result {final_count:,} is >25% "
        f"below the current delivered {prev_count:,}; refusing to overwrite. "
        f"Previous data is preserved."
    )

print(
    f"Guards passed: rows={final_count:,}, distinct(portfolio,CDI)={distinct_pc:,}, "
    f"recon_diff={recon_diff:.4f}, prev={prev_count}"
)

# ==================== Write ====================
# Overwrite (replaces TRUNCATE + INSERT from original)
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_fqn)
print(f"Written {final_count} rows to {target_fqn}")